In [1]:
import sys
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.13.9 2.0.2 2.3.3 1.6.1


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 2633

data = pd.read_csv("data/heart_disease.csv")

feature_names = [c for c in data.columns if c != "disease"]

X = data[feature_names].to_numpy()
y = data["disease"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=SEED
)

print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


 Part 1 — Entropy and Information Gain

1. Parent Entropy

We have 12 patients:
- 6 patients have disease = 1
- 6 patients have disease = 0

P(1) = 6/12 = 0.5
P(0) = 6/12 = 0.5

H = -0.5 * log2(0.5) - 0.5 * log2(0.5)

H = 1

The entropy is 1 because both classes have the same number of patients.

 2. Information Gain — age_55_plus

For age_55_plus = 1:
- 6 patients: 5 with disease and 1 without disease.
- H1 = -(5/6) × log2(5/6) - (1/6) × log2(1/6)
- H1 = 0.6500

For age_55_plus = 0:
- 6 patients: 1 with disease and 5 without disease.
- H0 = -(1/6) × log2(1/6) - (5/6) × log2(5/6)
- H0 = 0.6500

Weighted entropy:

H(children) = (6/12) × 0.6500 + (6/12) × 0.6500 = 0.6500

Information Gain:

IG = 1 - 0.6500 = **0.3500**

The age feature reduces the entropy, so it is useful for splitting the patients.

 Information Gain — exercise_angina

For exercise_angina = 1:
- 4 patients: 4 with disease, 0 without disease.
- H1 = 0

For exercise_angina = 0:
- 8 patients: 2 with disease, 6 without disease.
- H0 = -(2/8) * log2(2/8) - (6/8) * log2(6/8)
- H0 = 0.8113

Weighted entropy:

H(children) = (4/12) * 0 + (8/12) * 0.8113
H(children) = 0.5409

Information Gain:

IG = 1 - 0.5409 = 0.4591

The information gain is higher than for age_55_plus.
This feature gives a better split.

 Information Gain — male

For male = 1:
- 6 patients: 3 with disease, 3 without disease.
- H1 = -(3/6) * log2(3/6) - (3/6) * log2(3/6) = 1

For male = 0:
- 6 patients: 3 with disease, 3 without disease.
- H0 = 1

Weighted entropy:
H(children) = (6/12) * 1 + (6/12) * 1 = 1

IG(male) = 1 - 1 = 0

 Information Gain — high_bp

For high_bp = 1:
- 6 patients: 4 with disease, 2 without disease.
- H1 = -(4/6) * log2(4/6) - (2/6) * log2(2/6) = 0.9183

For high_bp = 0:
- 6 patients: 2 with disease, 4 without disease.
- H0 = 0.9183

Weighted entropy:
H(children) = (6/12) * 0.9183 + (6/12) * 0.9183 = 0.9183

IG(high_bp) = 1 - 0.9183 = 0.0817

 Root Selection

The information gains are:

- age_55_plus = 0.3500
- exercise_angina = 0.4591
- male = 0.0000
- high_bp = 0.0817

I choose exercise_angina as the root because it has the highest information gain.

 Question 3 — One Level Down

After the root split, exercise_angina = 1 is pure.

For exercise_angina = 0, we have 8 patients:
- 2 with disease
- 6 without disease

Parent entropy = 0.8113

**age_55_plus:**

IG = 0.8113 - [(2/8) * 1 + (6/8) * 0.6500]

IG = 0.0738

**male:**

IG = 0.8113 - [(3/8) * 0 + (5/8) * 0.9710]

IG = 0.2044

**high_bp:**

IG = 0.8113 - [(3/8) * 0.9183 + (5/8) * 0.7219]

IG = 0.0157

**Conclusion:**

I choose male because it has the highest information gain in this branch.

At the root, male had zero information gain. But after the first split, the data changed. Now male can separate patients better than the other features.

Part 2 — Decision Tree from Scratch

In [3]:
def entropy(y):
    """Calculate entropy of class labels."""
    counts = np.bincount(y)
    probabilities = counts[counts > 0] / len(y)
    return -np.sum(probabilities * np.log2(probabilities))

In [4]:
print(entropy(np.array([1, 1, 0, 0])))
print(entropy(np.array([1, 1, 1, 1])))
print(entropy(np.array([1, 1, 1, 0])))

1.0
-0.0
0.8112781244591328


In [5]:
def information_gain(y, left):
    """Calculate information gain after splitting."""
    y_left = y[left]
    y_right = y[~left]

    n = len(y)

    weighted_entropy = (
        len(y_left) / n * entropy(y_left)
        + len(y_right) / n * entropy(y_right)
    )

    return entropy(y) - weighted_entropy

In [6]:
y = np.array([1, 1, 0, 0])
left = np.array([True, True, False, False])

print(information_gain(y, left))

1.0


In [7]:
def best_split(X, y, min_samples_leaf=1):
    """Find the feature and threshold with the highest gain."""
    best_gain = 0
    best_result = None

    for feature in range(X.shape[1]):
        values = np.unique(X[:, feature])
        thresholds = (values[:-1] + values[1:]) / 2

        for threshold in thresholds:
            left = X[:, feature] <= threshold

            if left.sum() < min_samples_leaf:
                continue

            if (~left).sum() < min_samples_leaf:
                continue

            gain = information_gain(y, left)

            if gain > best_gain + 1e-12:
                best_gain = gain
                best_result = (feature, threshold, gain)

    return best_result

In [8]:
X_example = np.array([[25], [30], [50], [60]])
y_example = np.array([0, 0, 1, 1])

print(best_split(X_example, y_example))

(0, np.float64(40.0), np.float64(1.0))


In [9]:
from sklearn.tree import DecisionTreeClassifier

our_split = best_split(X_train, y_train)

ref = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=1,
    random_state=SEED
)

ref.fit(X_train, y_train)

print("Our split:", feature_names[our_split[0]], our_split[1])
print("Sklearn split:", feature_names[ref.tree_.feature[0]], ref.tree_.threshold[0])

Our split: thal 4.5
Sklearn split: thal 4.5
